# AE3: transformación y enriquecimiento de datos

**Proyecto:** perfil de barrios para arrendar o comprar vivienda en Colombia.
**Dataset:** [`aptologia/arriendo-rentabilidad-por-barrio-colombia-2026`](https://huggingface.co/datasets/aptologia/arriendo-rentabilidad-por-barrio-colombia-2026) (Aptología, CC BY 4.0), el mismo de AE2.

**Flujo de este notebook**

```
1. Preguntas y carga  →  2. Transformación y validación  →  3. Enriquecimiento
→  4. Respuestas (una tabla por pregunta)  →  5. Exportación (.csv y .txt) y verificación
```

**Salidas que genera (rutas relativas a la raíz del repositorio)**

| Archivo | Contenido |
|---|---|
| `AE3/datos/dataset_preparado.csv` | DataFrame final transformado y enriquecido (979 barrios) |
| `AE3/resultados/tablas/*.csv` | Una tabla por pregunta de negocio y tablas de control |
| `AE3/resultados/resumen_dataframe_final.txt` | Descripción del DataFrame final, regenerada en cada ejecución |

## 1. Necesidad, preguntas de negocio y variables

**Necesidad (AE2):** las personas que buscan vivienda no tienen una referencia sencilla de cuánto cuesta cada barrio ni de cuánta oferta hay en arriendo y en venta. El proyecto construye un informe para perfilar barrios y presupuestar el arriendo o la compra de un inmueble, aplicable a varias ciudades de Colombia.

| # | Pregunta de negocio | Variables del dataset | Transformación / enriquecimiento necesario | Resultado esperado |
|---|---|---|---|---|
| P1 | ¿Cuánto cuesta arrendar y comprar en cada barrio? | `ciudad`, `barrio`, `mediana_arriendo`, `mediana_venta`, `ok_todos` | Precios en millones de COP; arriendo anual; rango del barrio dentro de su ciudad; banda de presupuesto | `p1_costo_por_barrio.csv` |
| P2 | ¿Cuántas ofertas de arriendo y de venta tiene cada barrio y cuál predomina? | `avisos_arriendo`, `avisos_venta` | Total de avisos; % de avisos en arriendo; predominio de la oferta | `p2_ofertas_por_barrio.csv`, `p2_predominio_por_ciudad.csv` |
| P3 | ¿Qué barrios concentran más oferta en cada ciudad? | `avisos_arriendo`, `avisos_venta`, `ciudad` | Participación del barrio en la oferta de su ciudad; puesto; participación acumulada | `p3_top10_oferta_por_ciudad.csv` |
| P4 | ¿En qué barrios conviene más arrendar o comprar? | `mediana_2_arriendo`, `mediana_2_venta`, `rent_bruta_2alc_pct_calc`, `anios_arriendo_2alc_calc`, `ok_2alc` | Nivel de rentabilidad; marca de valor atípico | `p4_arrendar_vs_comprar.csv` |
| P5 | ¿Cómo se comparan las ciudades en costo y oferta? | Todas las anteriores, agregadas por `ciudad` | Medianas entre barrios confiables; participación nacional; concentración | `p5_resumen_por_ciudad.csv` |

**Ajustes al alcance definido en AE2 (justificados)**

1. **Mediana en lugar de promedio.** El objetivo de AE2 decía "promedio de costo", pero el dataset publica la **mediana** de los avisos de cada barrio, no los avisos individuales; no es posible calcular un promedio. La mediana es además más adecuada para precios de vivienda porque no la distorsionan unos pocos inmuebles de lujo.
2. **Se agregan P4 y P5.** El objetivo general habla de presupuestar "el valor de hacer el negocio de venta o alquiler" y de aplicarlo "a otras ciudades". P4 responde lo primero con la rentabilidad bruta que trae el dataset y P5 lo segundo.
3. **Se conservan los 979 barrios**, aunque los rankings usan solo barrios con muestra suficiente, porque el objetivo 2 pide poder consultar cualquier barrio.

**Limitaciones conocidas desde el inicio:** son precios **pedidos** en avisos (no precios de cierre), de una sola fecha (5 de octubre de 2026), agregados por barrio y solo de apartamentos. No hay área (m²), estrato ni costos de administración, por lo que no se puede calcular precio por m² ni rentabilidad neta.

In [1]:
import sys  # Permite agregar carpetas a la ruta de búsqueda de módulos
from datetime import datetime  # Para registrar cuándo se generan los archivos
from pathlib import Path  # Manejo de rutas de carpetas y archivos

ROOT = Path.cwd().resolve()  # Carpeta desde donde se ejecuta el notebook
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():  # Subimos hasta encontrar la raíz del repositorio
    ROOT = ROOT.parent  # Una carpeta hacia arriba

sys.path.insert(0, str(ROOT / "src"))  # Clases reutilizables del curso (src/clases_po)
sys.path.insert(0, str(ROOT / "AE3" / "src"))  # Funciones de apoyo de esta entrega (AE3/src)

import numpy as np  # Cálculos numéricos y condiciones vectorizadas
import pandas as pd  # Manejo de DataFrames
from clases_po import HFDatasetLoader, Reportes  # Carga desde Hugging Face y reporte .txt (Triple A)
from utilidades_ae3 import (  # Funciones propias de la AE3
    BitacoraCalidad,  # Registro de controles de calidad
    comparar_estructura,  # Tabla de cambios por columna (antes vs. después)
    exportar_csv_verificado,  # Exporta un CSV y comprueba que coincida con la tabla
    normalizar_texto,  # Minúsculas, sin tildes ni espacios repetidos
    rango_relativo_en_ciudad,  # Cuartil del barrio dentro de su ciudad
    resumen_dimensiones,  # Filas, columnas y nulos antes vs. después
)

pd.set_option("display.max_columns", 40)  # Mostrar hasta 40 columnas
pd.set_option("display.width", 220)  # Ancho de impresión de tablas

CARPETA_AE3 = ROOT / "AE3"  # Carpeta de esta entrega
RUTA_DATASET = CARPETA_AE3 / "datos" / "dataset_preparado.csv"  # Dataset preparado
CARPETA_TABLAS = CARPETA_AE3 / "resultados" / "tablas"  # Tablas de resultados
RUTA_TXT = CARPETA_AE3 / "resultados" / "resumen_dataframe_final.txt"  # Resumen descriptivo
print("Repositorio:", ROOT.name)  # Confirmamos la raíz (solo el nombre, sin rutas personales)

Repositorio: Ideacion


### 1.1 Carga del dataset y copia sin modificar

Se usa la clase `HFDatasetLoader` de `src/clases_po`. Se fija la **revisión** (versión exacta) del dataset para que cualquier persona que ejecute el notebook obtenga los mismos datos y las mismas cifras, aunque el autor publique una versión nueva. `df_original` se conserva sin cambios para comparar al final.

In [2]:
DATASET_ID = "aptologia/arriendo-rentabilidad-por-barrio-colombia-2026"  # Identificador en Hugging Face
CONFIG = "rentabilidad_arriendo_venta_barrio"  # Configuración (subconjunto) usada
SPLIT = "train"  # Única división disponible
REVISION = "b752ea67e4c2615a723900d7654e3553e6a546c1"  # Versión exacta del dataset (la misma de AE2)

loader = HFDatasetLoader(dataset_name=DATASET_ID, config=CONFIG, split=SPLIT, revision=REVISION)  # Configuramos la carga
df_original = loader.load()  # Descargamos el dataset completo (sin muestra)
df = df_original.copy()  # Copia de trabajo: todas las transformaciones se hacen sobre df
print(loader.info())  # Dataset, configuración, split, revisión, filas y columnas
df_original.head()  # Primeras filas tal como vienen de la fuente

Generating train split: 100%|██████████| 979/979
{'dataset': 'aptologia/arriendo-rentabilidad-por-barrio-colombia-2026', 'config': 'rentabilidad_arriendo_venta_barrio', 'split': 'train', 'revision': 'b752ea67e4c2615a723900d7654e3553e6a546c1', 'filas': 979, 'columnas': 19}


,fecha_arriendo,fecha_venta,ciudad_id,ciudad,barrio_id,barrio,avisos_arriendo,mediana_arriendo,avisos_venta,mediana_venta,ok_todos,rent_bruta_todos_pct_calc,avisos_2_arriendo,mediana_2_arriendo,avisos_2_venta,mediana_2_venta,ok_2alc,rent_bruta_2alc_pct_calc,anios_arriendo_2alc_calc
0,2026-10-05,2026-10-05,medellin,Medellín,laureles,Laureles,6722,4000000,4067,791000000,True,6.068268,1385,3400000.0,441,6.250000e+08,True,6.528000,15.318627
1,2026-10-05,2026-10-05,medellin,Medellín,castropol,Castropol,2788,5350000,666,980000000,True,6.551020,828,4550000.0,140,9.375000e+08,True,5.824000,17.170330
2,2026-10-05,2026-10-05,medellin,Medellín,el-tesoro,El Tesoro,2680,7500000,719,1300000000,True,6.923077,545,6600000.0,140,1.160000e+09,True,6.827586,14.646465
3,2026-10-05,2026-10-05,medellin,Medellín,santa-maria-de-los-angeles,Santa Maria de los Angeles,2321,5500000,649,1050000000,True,6.285714,679,4700000.0,93,7.950000e+08,True,7.094340,14.095745
4,2026-10-05,2026-10-05,medellin,Medellín,calasanz,Calasanz,2281,2600000,968,440000000,True,7.090909,788,2400000.0,293,3.550000e+08,True,8.112676,12.326389


## 2. Transformación y validación

Cada control sigue el mismo procedimiento: **(1)** se define la regla, **(2)** se cuenta cuántos registros la incumplen o se ven afectados, **(3)** se decide el tratamiento y se justifica, y **(4)** se registra en la bitácora de calidad. Ningún registro se elimina ni se rellena sin una regla explícita.

In [3]:
bitacora = BitacoraCalidad()  # Bitácora vacía donde se registran todos los controles

### 2.1 Tipos de datos y fechas

In [4]:
for col in ["fecha_arriendo", "fecha_venta"]:  # Las fechas llegan como texto "AAAA-MM-DD"
    df[col] = pd.to_datetime(df[col], format="%Y-%m-%d", errors="coerce")  # Texto -> fecha; inválidas quedan como NaT

fechas_invalidas = int(df[["fecha_arriendo", "fecha_venta"]].isna().sum().sum())  # Fechas que no se pudieron convertir
fechas_distintas = int((df["fecha_arriendo"] != df["fecha_venta"]).sum())  # Filas donde las dos fechas difieren
print("Fechas inválidas:", fechas_invalidas)  # Esperado: 0
print("Filas con fecha de arriendo distinta a la de venta:", fechas_distintas)  # Esperado: 0
print("Fechas presentes:", df["fecha_arriendo"].dt.date.unique())  # Fecha(s) de corte del dataset

bitacora.registrar("Tipo de fechas", "Convertir fecha_arriendo y fecha_venta de texto a fecha (AAAA-MM-DD)",
                   len(df), f"Convertidas las {len(df)} filas; {fechas_invalidas} fechas inválidas.")
df["fecha_corte"] = df["fecha_arriendo"]  # Una sola columna de fecha, porque ambas son iguales
df = df.drop(columns=["fecha_arriendo", "fecha_venta"])  # Quitamos las dos columnas redundantes
bitacora.registrar("Fechas redundantes", "fecha_arriendo debe ser igual a fecha_venta", fechas_distintas,
                   "Son idénticas en todas las filas: se unifican en fecha_corte y se eliminan las dos originales.")

numericas = ["avisos_arriendo", "mediana_arriendo", "avisos_venta", "mediana_venta", "rent_bruta_todos_pct_calc",
             "avisos_2_arriendo", "mediana_2_arriendo", "avisos_2_venta", "mediana_2_venta",
             "rent_bruta_2alc_pct_calc", "anios_arriendo_2alc_calc"]  # Columnas que deben ser numéricas
no_numericas = [c for c in numericas if not pd.api.types.is_numeric_dtype(df[c])]  # Las que no lo sean
no_logicas = [c for c in ["ok_todos", "ok_2alc"] if df[c].dtype != bool]  # Indicadores que no sean True/False
print("Columnas numéricas con tipo incorrecto:", no_numericas)  # Esperado: []
print("Indicadores con tipo incorrecto:", no_logicas)  # Esperado: []
bitacora.registrar("Tipos numéricos y lógicos", "Avisos y precios numéricos; ok_todos y ok_2alc lógicos (True/False)",
                   len(no_numericas) + len(no_logicas), "Sin problemas: los tipos ya son correctos, no se modifican.")

Fechas inválidas: 0
Filas con fecha de arriendo distinta a la de venta: 0
Fechas presentes: [datetime.date(2026, 10, 5)]
Columnas numéricas con tipo incorrecto: []
Indicadores con tipo incorrecto: []


### 2.2 Duplicados

In [5]:
dup_filas = int(df.duplicated().sum())  # Filas idénticas en todas las columnas
dup_clave = int(df.duplicated(["ciudad_id", "barrio_id"]).sum())  # Mismo barrio repetido en la misma ciudad
ids_repetidos = int(df["barrio_id"].duplicated().sum())  # Mismo identificador de barrio en ciudades distintas
print("Filas duplicadas completas:", dup_filas)  # Esperado: 0
print("Duplicados por clave ciudad + barrio:", dup_clave)  # Esperado: 0
print("barrio_id repetidos entre ciudades:", ids_repetidos)  # Ej.: "santa-ana" en Bogotá y en Bucaramanga
print(df[df["barrio_id"].duplicated(keep=False)].sort_values("barrio_id")[["barrio_id", "ciudad", "barrio"]].head(6))

bitacora.registrar("Duplicados", "Filas idénticas y clave ciudad_id + barrio_id", dup_filas + dup_clave,
                   "No hay duplicados; no se elimina ningún registro.")
bitacora.registrar("Clave de identificación", "barrio_id debe ser único", ids_repetidos,
                   "barrio_id se repite entre ciudades (barrios homónimos, no duplicados). Se define la clave compuesta ciudad_id + barrio_id.")

Filas duplicadas completas: 0
Duplicados por clave ciudad + barrio: 0
barrio_id repetidos entre ciudades: 85
       barrio_id    ciudad       barrio
954  20-de-julio      Chía  20 de Julio
487  20-de-julio    Bogotá  20 de Julio
702      alameda      Cali      Alameda
406      alameda    Bogotá      Alameda
112       alcala  Medellín       Alcala
497       alcala    Bogotá       Alcala


### 2.3 Inconsistencias de texto y normalización de categorías

In [6]:
cambios_texto = 0  # Contador de valores modificados
for col in ["ciudad", "barrio"]:  # Columnas de texto visibles para el usuario
    limpio = df[col].str.strip().str.replace(r"\s+", " ", regex=True)  # Quita espacios al inicio/fin y espacios dobles
    cambios_texto += int((limpio != df[col]).sum())  # Cuántos valores cambian
    df[col] = limpio  # Guardamos el texto limpio

pares = df[["ciudad_id", "ciudad"]].drop_duplicates()  # Combinaciones únicas id - nombre de ciudad
ciudad_inconsistente = int(pares["ciudad_id"].duplicated().sum() + pares["ciudad"].duplicated().sum())  # Debe ser 1 a 1
df["barrio_clave"] = df["barrio"].map(normalizar_texto)  # Nombre normalizado: minúsculas y sin tildes
dup_nombre = int(df.duplicated(["ciudad_id", "barrio_clave"]).sum())  # Mismo barrio escrito de dos formas
print("Valores de texto corregidos (espacios):", cambios_texto)  # Esperado: 0
print("Ciudades con más de un nombre o id:", ciudad_inconsistente)  # Esperado: 0
print("Barrios repetidos al normalizar el nombre:", dup_nombre)  # Esperado: 0
print("Ciudades:", sorted(df["ciudad"].unique()))  # Lista de categorías de ciudad

bitacora.registrar("Espacios en texto", "Sin espacios al inicio/fin ni dobles en ciudad y barrio", cambios_texto,
                   "Sin hallazgos; se aplica la limpieza igualmente para garantizar el formato.")
bitacora.registrar("Categorías de ciudad", "Cada ciudad_id corresponde a un solo nombre de ciudad", ciudad_inconsistente,
                   "Relación 1 a 1 correcta; 13 ciudades.")
bitacora.registrar("Nombres de barrio equivalentes", "Sin barrios repetidos al comparar nombre sin tildes ni mayúsculas",
                   dup_nombre, "Sin hallazgos. Se crea barrio_clave (normalizado) para búsquedas y filtros.")

Valores de texto corregidos (espacios): 0
Ciudades con más de un nombre o id: 0
Barrios repetidos al normalizar el nombre: 0
Ciudades: ['Barranquilla', 'Bogotá', 'Bucaramanga', 'Cali', 'Cartagena', 'Chía', 'Cúcuta', 'Madrid', 'Manizales', 'Medellín', 'Pereira', 'Rionegro', 'Soacha']


### 2.4 Rangos válidos de los valores

In [7]:
reglas_rango = {  # Cada regla devuelve True en las filas que la INCUMPLEN
    "avisos de arriendo y venta >= 1": (df["avisos_arriendo"] < 1) | (df["avisos_venta"] < 1),
    "medianas de precio > 0": (df["mediana_arriendo"] <= 0) | (df["mediana_venta"] <= 0),
    "arriendo dentro del rango sano de la fuente ($200.000–$30.000.000)": ~df["mediana_arriendo"].between(200_000, 30_000_000),
    "avisos de 2 alcobas <= avisos totales": (df["avisos_2_arriendo"] > df["avisos_arriendo"]) | (df["avisos_2_venta"] > df["avisos_venta"]),
}
for regla, incumple in reglas_rango.items():  # Revisamos cada regla
    print(f"{regla}: {int(incumple.sum())} filas incumplen")  # Esperado: 0 en todas
total_fuera_rango = int(sum(int(m.sum()) for m in reglas_rango.values()))  # Total de incumplimientos
print("Rango de avisos de arriendo:", df["avisos_arriendo"].min(), "a", df["avisos_arriendo"].max())  # Mínimo y máximo
print("Rango de arriendo (COP/mes):", f"{df['mediana_arriendo'].min():,} a {df['mediana_arriendo'].max():,}")  # Mínimo y máximo
bitacora.registrar("Rangos válidos", "; ".join(reglas_rango), total_fuera_rango,
                   "Todos los valores están en rangos válidos; no se modifica ningún registro.")

avisos de arriendo y venta >= 1: 0 filas incumplen
medianas de precio > 0: 0 filas incumplen
arriendo dentro del rango sano de la fuente ($200.000–$30.000.000): 0 filas incumplen
avisos de 2 alcobas <= avisos totales: 0 filas incumplen
Rango de avisos de arriendo: 15 a 6722
Rango de arriendo (COP/mes): 550,000 a 20,500,000


### 2.5 Coherencia de cálculos y de las reglas de calidad de la fuente
Se recalculan las columnas derivadas y se verifica la regla que define cada indicador, para entender su significado (la ficha del dataset remite al PDF del autor).

In [8]:
rent_calc = df["mediana_arriendo"] * 12 / df["mediana_venta"] * 100  # Rentabilidad bruta = arriendo anual / precio de venta
rent2_calc = df["mediana_2_arriendo"] * 12 / df["mediana_2_venta"] * 100  # Lo mismo para 2 alcobas
anios_calc = df["mediana_2_venta"] / (df["mediana_2_arriendo"] * 12)  # Años de arriendo que equivalen al precio de compra
regla_ok_todos = (df["avisos_arriendo"] >= 30) & (df["avisos_venta"] >= 30)  # Hipótesis: >= 30 avisos en cada operación
regla_ok_2alc = (df["avisos_2_arriendo"] >= 20) & (df["avisos_2_venta"] >= 20)  # Hipótesis: >= 20 avisos de 2 alcobas (ficha)

coherencia = {  # Cantidad de filas donde el cálculo o la regla NO coincide
    "rent_bruta_todos_pct_calc = arriendo x 12 / venta x 100": int(((rent_calc - df["rent_bruta_todos_pct_calc"]).abs() > 1e-6).sum()),
    "rent_bruta_2alc_pct_calc = arriendo 2alc x 12 / venta 2alc x 100": int(((rent2_calc - df["rent_bruta_2alc_pct_calc"]).abs() > 1e-6).sum()),
    "anios_arriendo_2alc_calc = venta 2alc / (arriendo 2alc x 12)": int(((anios_calc - df["anios_arriendo_2alc_calc"]).abs() > 1e-6).sum()),
    "ok_todos = (avisos arriendo >= 30) y (avisos venta >= 30)": int((regla_ok_todos != df["ok_todos"]).sum()),
    "ok_2alc = (avisos 2alc arriendo >= 20) y (avisos 2alc venta >= 20)": int((regla_ok_2alc != df["ok_2alc"]).sum()),
}
for regla, n in coherencia.items():  # Mostramos el resultado de cada verificación
    print(f"{regla}: {n} diferencias")  # Esperado: 0
print("Barrios con muestra suficiente (ok_todos):", int(df["ok_todos"].sum()), "de", len(df))  # Barrios confiables
print("Barrios comparables en 2 alcobas (ok_2alc):", int(df["ok_2alc"].sum()), "de", len(df))  # Barrios para P4
bitacora.registrar("Coherencia de cálculos", "Recalcular rentabilidades y años de arriendo", sum(list(coherencia.values())[:3]),
                   "Coinciden en todas las filas: se confirma el significado de las columnas calculadas.")
bitacora.registrar("Reglas de calidad de la fuente", "ok_todos: >=30 avisos en arriendo y venta; ok_2alc: >=20 avisos de 2 alcobas en ambos",
                   sum(list(coherencia.values())[3:]), "Las reglas se cumplen exactamente; se usan para definir la confianza de cada barrio.")

rent_bruta_todos_pct_calc = arriendo x 12 / venta x 100: 0 diferencias
rent_bruta_2alc_pct_calc = arriendo 2alc x 12 / venta 2alc x 100: 0 diferencias
anios_arriendo_2alc_calc = venta 2alc / (arriendo 2alc x 12): 0 diferencias
ok_todos = (avisos arriendo >= 30) y (avisos venta >= 30): 0 diferencias
ok_2alc = (avisos 2alc arriendo >= 20) y (avisos 2alc venta >= 20): 0 diferencias
Barrios con muestra suficiente (ok_todos): 620 de 979
Barrios comparables en 2 alcobas (ok_2alc): 308 de 979


### 2.6 Valores faltantes

In [9]:
nulos = df.isna().sum()  # Nulos por columna
nulos = nulos[nulos > 0].to_frame("nulos").assign(porcentaje=lambda t: (t["nulos"] / len(df) * 100).round(1))  # Solo columnas con nulos
print(nulos)  # Tabla de nulos

nulo_arr_explicado = int((df["mediana_2_arriendo"].isna() == (df["avisos_2_arriendo"] < 8)).all())  # ¿Nulo exactamente cuando hay < 8 avisos?
nulo_ven_explicado = int((df["mediana_2_venta"].isna() == (df["avisos_2_venta"] < 8)).all())  # Igual para venta
nulo_rent_explicado = int((df["rent_bruta_2alc_pct_calc"].isna() == ~df["ok_2alc"]).all())  # ¿Nulo exactamente cuando no es comparable?
print("mediana_2_arriendo es nula solo cuando hay menos de 8 avisos de 2 alcobas:", bool(nulo_arr_explicado))
print("mediana_2_venta es nula solo cuando hay menos de 8 avisos de 2 alcobas:", bool(nulo_ven_explicado))
print("Rentabilidad y años de 2 alcobas son nulos solo cuando ok_2alc es False:", bool(nulo_rent_explicado))

filas_con_nulos = int(df.isna().any(axis=1).sum())  # Filas con al menos un valor faltante
print("Filas con al menos un nulo:", filas_con_nulos, "| nulos totales:", int(nulos["nulos"].sum()))
bitacora.registrar("Valores faltantes", "Nulos en columnas de 2 alcobas (mediana < 8 avisos; rentabilidad si ok_2alc = False)",
                   filas_con_nulos,
                   "Son nulos estructurales (muestra insuficiente), no errores. NO se imputan: rellenarlos inventaría precios. "
                   "Se dejan vacíos y P4 usa solo los 308 barrios comparables.")

                          nulos  porcentaje
mediana_2_arriendo          231        23.6
mediana_2_venta             287        29.3
rent_bruta_2alc_pct_calc    671        68.5
anios_arriendo_2alc_calc    671        68.5
mediana_2_arriendo es nula solo cuando hay menos de 8 avisos de 2 alcobas: True
mediana_2_venta es nula solo cuando hay menos de 8 avisos de 2 alcobas: True
Rentabilidad y años de 2 alcobas son nulos solo cuando ok_2alc es False: True
Filas con al menos un nulo: 671 | nulos totales: 1860


### 2.7 Valores atípicos de rentabilidad
Regla de Tukey: es atípico un valor por debajo de Q1 − 1,5×RIC o por encima de Q3 + 1,5×RIC (RIC = Q3 − Q1).

In [10]:
def marcar_atipicos(serie: pd.Series) -> pd.Series:  # Devuelve True en los valores atípicos (regla de Tukey)
    q1, q3 = serie.quantile([0.25, 0.75])  # Primer y tercer cuartil
    ric = q3 - q1  # Rango intercuartílico
    return (serie < q1 - 1.5 * ric) | (serie > q3 + 1.5 * ric)  # Fuera de los límites -> atípico

df["rentabilidad_atipica"] = marcar_atipicos(df["rent_bruta_todos_pct_calc"])  # Sobre los 979 barrios
comparables = df["ok_2alc"]  # Barrios comparables en 2 alcobas
df["rentabilidad_2alc_atipica"] = False  # Por defecto no atípico
df.loc[comparables, "rentabilidad_2alc_atipica"] = marcar_atipicos(df.loc[comparables, "rent_bruta_2alc_pct_calc"])  # Solo entre comparables

n_atip = int(df["rentabilidad_atipica"].sum())  # Cantidad de atípicos (todos los apartamentos)
n_atip2 = int(df["rentabilidad_2alc_atipica"].sum())  # Cantidad de atípicos (2 alcobas)
print("Rentabilidad atípica (todos):", n_atip, "barrios;", int((df["rentabilidad_atipica"] & df["ok_todos"]).sum()), "de ellos con muestra suficiente")
print("Rentabilidad atípica (2 alcobas, entre comparables):", n_atip2, "barrios")
print(df.loc[df["rentabilidad_atipica"], ["ciudad", "barrio", "mediana_arriendo", "mediana_venta", "rent_bruta_todos_pct_calc", "ok_todos"]]
      .sort_values("rent_bruta_todos_pct_calc").head(5))  # Ejemplos de valores bajos
filas_atipicas = int((df["rentabilidad_atipica"] | df["rentabilidad_2alc_atipica"]).sum())  # Barrios con al menos una marca
print("Barrios con al menos una rentabilidad atípica:", filas_atipicas)
bitacora.registrar("Valores atípicos", "Regla de Tukey (1,5 x RIC) sobre la rentabilidad bruta (todos y 2 alcobas)", filas_atipicas,
                   "Se MARCAN, no se eliminan: son medianas reales de avisos (p. ej. barrios céntricos con precios de venta altos). "
                   "Se identifican en las tablas para interpretarlos con cautela.")

Rentabilidad atípica (todos): 42 barrios; 22 de ellos con muestra suficiente
Rentabilidad atípica (2 alcobas, entre comparables): 24 barrios
        ciudad            barrio  mediana_arriendo  mediana_venta  rent_bruta_todos_pct_calc  ok_todos
472     Bogotá         El Centro           1700000     2200000000                   0.927273      True
240   Medellín       Santa Elena           2775000     1191030000                   2.795899      True
717       Cali            Obrero            780000      330048140                   2.835950      True
932  Cartagena        Manzanillo           2540000     1063965000                   2.864756      True
636     Bogotá  Benjamin Herrera           1580000      650000000                   2.916923     False
Barrios con al menos una rentabilidad atípica: 60


### 2.8 Unidades y nombres de columnas
Los precios pasan de pesos a **millones de pesos (COP)** para leerlos fácilmente, y cada columna lleva su unidad en el nombre.

In [11]:
precios = ["mediana_arriendo", "mediana_venta", "mediana_2_arriendo", "mediana_2_venta"]  # Columnas de precio en COP
valores_convertidos = int(df[precios].notna().sum().sum())  # Valores no nulos que se van a convertir
df["mediana_arriendo"] = df["mediana_arriendo"] / 1_000_000  # COP/mes -> millones COP/mes
df["mediana_venta"] = df["mediana_venta"] / 1_000_000  # COP -> millones COP
df["mediana_2_arriendo"] = df["mediana_2_arriendo"] / 1_000_000  # Igual para 2 alcobas (los nulos se mantienen)
df["mediana_2_venta"] = df["mediana_2_venta"] / 1_000_000  # Igual para 2 alcobas
nombres = {  # Nombre original -> nombre claro con unidad
    "mediana_arriendo": "arriendo_mediano_mill_cop_mes",
    "mediana_venta": "venta_mediana_mill_cop",
    "avisos_2_arriendo": "avisos_2alc_arriendo",
    "avisos_2_venta": "avisos_2alc_venta",
    "mediana_2_arriendo": "arriendo_2alc_mill_cop_mes",
    "mediana_2_venta": "venta_2alc_mill_cop",
    "rent_bruta_todos_pct_calc": "rentabilidad_bruta_pct",
    "rent_bruta_2alc_pct_calc": "rentabilidad_bruta_2alc_pct",
    "anios_arriendo_2alc_calc": "anios_arriendo_2alc",
    "ok_todos": "muestra_suficiente",
    "ok_2alc": "comparable_2alc",
}
df = df.rename(columns=nombres)  # Aplicamos los nombres nuevos
bitacora.registrar("Unidades y nombres", "Precios de COP a millones de COP; nombres con unidad", len(df),
                   f"{valores_convertidos} valores de precio convertidos en 4 columnas de las {len(df)} filas (los nulos siguen nulos); "
                   f"{len(nombres)} columnas renombradas.")
print("Valores de precio convertidos:", valores_convertidos)
df[["ciudad", "barrio", "arriendo_mediano_mill_cop_mes", "venta_mediana_mill_cop", "rentabilidad_bruta_pct"]].head()  # Vista

Valores de precio convertidos: 3398


,ciudad,barrio,arriendo_mediano_mill_cop_mes,venta_mediana_mill_cop,rentabilidad_bruta_pct
0,Medellín,Laureles,4.00,791.0,6.068268
1,Medellín,Castropol,5.35,980.0,6.551020
2,Medellín,El Tesoro,7.50,1300.0,6.923077
3,Medellín,Santa Maria de los Angeles,5.50,1050.0,6.285714
4,Medellín,Calasanz,2.60,440.0,7.090909


### 2.9 Selección de columnas y registros

In [12]:
columnas_transformadas = [  # Columnas pertinentes al alcance, en orden lógico
    "fecha_corte", "ciudad_id", "ciudad", "barrio_id", "barrio", "barrio_clave",  # Identificación
    "avisos_arriendo", "avisos_venta", "arriendo_mediano_mill_cop_mes", "venta_mediana_mill_cop",  # Oferta y costo (P1-P3)
    "rentabilidad_bruta_pct", "muestra_suficiente", "rentabilidad_atipica",  # Rentabilidad y calidad
    "avisos_2alc_arriendo", "avisos_2alc_venta", "arriendo_2alc_mill_cop_mes", "venta_2alc_mill_cop",  # 2 alcobas (P4)
    "rentabilidad_bruta_2alc_pct", "anios_arriendo_2alc", "comparable_2alc", "rentabilidad_2alc_atipica",
]
df_transformado = df[columnas_transformadas].copy()  # DataFrame transformado (antes del enriquecimiento)
print("Columnas no seleccionadas:", [c for c in df.columns if c not in columnas_transformadas])  # Esperado: []
print("Filas conservadas:", len(df_transformado), "de", len(df_original))  # No se elimina ningún barrio
bitacora.registrar("Selección de registros", "Conservar todos los barrios; marcar los de muestra insuficiente", 0,
                   f"Se conservan los {len(df_transformado)} barrios para poder consultar cualquiera (objetivo 2). "
                   f"Los rankings usan solo los {int(df_transformado['muestra_suficiente'].sum())} con muestra suficiente.")

Columnas no seleccionadas: []
Filas conservadas: 979 de 979


### 2.10 Comparación antes y después de la transformación

In [13]:
display(resumen_dimensiones(df_original, df_transformado))  # Filas, columnas y nulos totales
comparar_estructura(df_original, df_transformado, nombres)  # Tipo y nulos de cada columna, siguiendo los renombres

,antes,despues,diferencia
filas,979,979,0
columnas,19,21,2
valores nulos (total),1860,1860,0


,columna_original,columna_final,tipo_antes,tipo_despues,nulos_antes,nulos_despues,estado
0,fecha_arriendo,—,str,—,0,<NA>,eliminada (reemplazada o redundante)
1,fecha_venta,—,str,—,0,<NA>,eliminada (reemplazada o redundante)
2,ciudad_id,ciudad_id,str,str,0,0,sin cambios
3,ciudad,ciudad,str,str,0,0,sin cambios
4,barrio_id,barrio_id,str,str,0,0,sin cambios
5,barrio,barrio,str,str,0,0,sin cambios
6,avisos_arriendo,avisos_arriendo,int64,int64,0,0,sin cambios
7,mediana_arriendo,arriendo_mediano_mill_cop_mes,int64,float64,0,0,"renombrada, tipo cambiado"
8,avisos_venta,avisos_venta,int64,int64,0,0,sin cambios
9,mediana_venta,venta_mediana_mill_cop,int64,float64,0,0,"renombrada, tipo cambiado"


Bitácora de calidad (se exporta como `AE3/resultados/tablas/control_bitacora_calidad.csv`):

In [14]:
tabla_bitacora = bitacora.tabla()  # Todos los controles aplicados
tabla_bitacora  # Mostramos la bitácora

,paso,control,regla,registros_afectados,decision
0,1,Tipo de fechas,Convertir fecha_arriendo y fecha_venta de text...,979,Convertidas las 979 filas; 0 fechas inválidas.
1,2,Fechas redundantes,fecha_arriendo debe ser igual a fecha_venta,0,Son idénticas en todas las filas: se unifican ...
2,3,Tipos numéricos y lógicos,Avisos y precios numéricos; ok_todos y ok_2alc...,0,"Sin problemas: los tipos ya son correctos, no ..."
3,4,Duplicados,Filas idénticas y clave ciudad_id + barrio_id,0,No hay duplicados; no se elimina ningún registro.
4,5,Clave de identificación,barrio_id debe ser único,85,barrio_id se repite entre ciudades (barrios ho...
5,6,Espacios en texto,Sin espacios al inicio/fin ni dobles en ciudad...,0,Sin hallazgos; se aplica la limpieza igualment...
6,7,Categorías de ciudad,Cada ciudad_id corresponde a un solo nombre de...,0,Relación 1 a 1 correcta; 13 ciudades.
7,8,Nombres de barrio equivalentes,Sin barrios repetidos al comparar nombre sin t...,0,Sin hallazgos. Se crea barrio_clave (normaliza...
8,9,Rangos válidos,avisos de arriendo y venta >= 1; medianas de p...,0,Todos los valores están en rangos válidos; no ...
9,10,Coherencia de cálculos,Recalcular rentabilidades y años de arriendo,0,Coinciden en todas las filas: se confirma el s...


## 3. Enriquecimiento

Variables calculadas solo con datos del propio dataset (no se combina otra tabla, porque el dataset ya contiene lo necesario para las cinco preguntas; así se evita el riesgo de multiplicar registros en una unión).

| Variable nueva | Regla o fórmula | Significado | Pregunta |
|---|---|---|---|
| `barrio_ciudad` | barrio + " (" + ciudad + ")" | Etiqueta única para tablas y gráficos | Todas |
| `periodo` | mes de `fecha_corte` (AAAA-MM) | Periodo de los datos | Todas |
| `arriendo_anual_mill_cop` | arriendo mensual × 12 | Costo de un año de arriendo | P1, P4 |
| `anios_arriendo_equivalentes` | venta ÷ arriendo anual | Años de arriendo que suman el precio de compra | P1, P4 |
| `rango_arriendo_ciudad`, `rango_venta_ciudad` | Cuartil del barrio dentro de su ciudad (solo ciudades con ≥ 8 barrios) | Qué tan caro es frente a su ciudad | P1 |
| `banda_presupuesto_arriendo` | Hasta 1,5 / 1,5–3 / 3–5 / más de 5 millones COP/mes | Rango de presupuesto mensual | P1 |
| `banda_presupuesto_venta` | Hasta 300 / 300–600 / 600–1.000 / más de 1.000 millones COP | Rango de presupuesto de compra | P1 |
| `total_avisos` | avisos de arriendo + avisos de venta | Oferta total del barrio | P2, P3 |
| `pct_avisos_arriendo` | avisos de arriendo ÷ total × 100 | Peso del arriendo en la oferta | P2 |
| `predominio_oferta` | Arriendo si % ≥ 60; Venta si % ≤ 40; si no, Equilibrado | Operación que predomina | P2 |
| `participacion_oferta_ciudad_pct` | total del barrio ÷ total de su ciudad × 100 | Peso del barrio en la oferta de su ciudad | P3 |
| `puesto_oferta_ciudad` | Posición por total de avisos dentro de la ciudad (1 = mayor) | Ranking de oferta | P3 |
| `nivel_confianza` | Alta si `muestra_suficiente`; si no, Baja | Confiabilidad del precio | Todas |
| `nivel_rentabilidad_2alc` | Baja < 6 %; Media 6–8 %; Alta ≥ 8 %; "No comparable" si no aplica | Atractivo del negocio de arrendar | P4 |

In [15]:
df_final = df_transformado.copy()  # Partimos del DataFrame transformado

df_final["barrio_ciudad"] = df_final["barrio"] + " (" + df_final["ciudad"] + ")"  # Etiqueta única
df_final["periodo"] = df_final["fecha_corte"].dt.to_period("M").astype(str)  # Periodo AAAA-MM

df_final["arriendo_anual_mill_cop"] = df_final["arriendo_mediano_mill_cop_mes"] * 12  # Arriendo de un año
df_final["anios_arriendo_equivalentes"] = df_final["venta_mediana_mill_cop"] / df_final["arriendo_anual_mill_cop"]  # Años para igualar la compra

df_final["rango_arriendo_ciudad"] = rango_relativo_en_ciudad(df_final, "arriendo_mediano_mill_cop_mes")  # Cuartil de arriendo en su ciudad
df_final["rango_venta_ciudad"] = rango_relativo_en_ciudad(df_final, "venta_mediana_mill_cop")  # Cuartil de venta en su ciudad

df_final["banda_presupuesto_arriendo"] = pd.cut(  # Bandas absolutas de presupuesto mensual
    df_final["arriendo_mediano_mill_cop_mes"], bins=[0, 1.5, 3, 5, np.inf],
    labels=["Hasta 1,5 M", "1,5 a 3 M", "3 a 5 M", "Más de 5 M"]).astype(str)
df_final["banda_presupuesto_venta"] = pd.cut(  # Bandas absolutas de presupuesto de compra
    df_final["venta_mediana_mill_cop"], bins=[0, 300, 600, 1000, np.inf],
    labels=["Hasta 300 M", "300 a 600 M", "600 a 1.000 M", "Más de 1.000 M"]).astype(str)

df_final["total_avisos"] = df_final["avisos_arriendo"] + df_final["avisos_venta"]  # Oferta total
df_final["pct_avisos_arriendo"] = df_final["avisos_arriendo"] / df_final["total_avisos"] * 100  # % de la oferta en arriendo
df_final["predominio_oferta"] = np.select(  # Operación predominante
    [df_final["pct_avisos_arriendo"] >= 60, df_final["pct_avisos_arriendo"] <= 40],
    ["Arriendo", "Venta"], default="Equilibrado")
total_ciudad = df_final.groupby("ciudad")["total_avisos"].transform("sum")  # Oferta total de la ciudad de cada fila
df_final["participacion_oferta_ciudad_pct"] = df_final["total_avisos"] / total_ciudad * 100  # Peso del barrio en su ciudad
df_final["puesto_oferta_ciudad"] = (df_final.groupby("ciudad")["total_avisos"]
                                    .rank(ascending=False, method="first").astype(int))  # 1 = barrio con más oferta

df_final["nivel_confianza"] = np.where(df_final["muestra_suficiente"], "Alta", "Baja")  # Confiabilidad del precio
r2 = df_final["rentabilidad_bruta_2alc_pct"]  # Rentabilidad de 2 alcobas (nula si no es comparable)
df_final["nivel_rentabilidad_2alc"] = np.select(  # Categoría de rentabilidad
    [r2.isna(), r2 < 6, r2 < 8], ["No comparable", "Baja (<6 %)", "Media (6-8 %)"], default="Alta (>=8 %)")

print("Columnas antes del enriquecimiento:", df_transformado.shape[1], "| después:", df_final.shape[1])  # Columnas agregadas
df_final[["barrio_ciudad", "arriendo_anual_mill_cop", "anios_arriendo_equivalentes", "rango_arriendo_ciudad",
          "banda_presupuesto_arriendo", "total_avisos", "predominio_oferta", "participacion_oferta_ciudad_pct"]].head()

Columnas antes del enriquecimiento: 21 | después: 36


,barrio_ciudad,arriendo_anual_mill_cop,anios_arriendo_equivalentes,rango_arriendo_ciudad,banda_presupuesto_arriendo,total_avisos,predominio_oferta,participacion_oferta_ciudad_pct
0,Laureles (Medellín),48.0,16.479167,Medio-alto,3 a 5 M,10789,Arriendo,8.157479
1,Castropol (Medellín),64.2,15.264798,Alto,Más de 5 M,3454,Arriendo,2.611543
2,El Tesoro (Medellín),90.0,14.444444,Alto,Más de 5 M,3399,Arriendo,2.569957
3,Santa Maria de los Angeles (Medellín),66.0,15.909091,Alto,Más de 5 M,2970,Arriendo,2.245594
4,Calasanz (Medellín),31.2,14.102564,Medio-bajo,"1,5 a 3 M",3249,Arriendo,2.456544


### 3.1 Validación del enriquecimiento

In [16]:
suma_part = df_final.groupby("ciudad")["participacion_oferta_ciudad_pct"].sum()  # Debe ser 100 en cada ciudad
assert np.allclose(suma_part, 100), suma_part  # Participaciones cuadran
assert (df_final["total_avisos"] == df_final["avisos_arriendo"] + df_final["avisos_venta"]).all()  # Total correcto
assert np.allclose(df_final["anios_arriendo_equivalentes"], 100 / df_final["rentabilidad_bruta_pct"])  # Años = 100 / rentabilidad
assert df_final.groupby("ciudad")["puesto_oferta_ciudad"].apply(lambda s: sorted(s) == list(range(1, len(s) + 1))).all()  # Puestos 1..n sin huecos
assert len(df_final) == len(df_original)  # El enriquecimiento no cambia el número de filas
assert not df_final.duplicated(["ciudad_id", "barrio_id"]).any()  # Sigue sin duplicados
print("Participación por ciudad (debe ser 100):", suma_part.round(6).unique())
print("Distribución de predominio:", df_final["predominio_oferta"].value_counts().to_dict())
print("Distribución de nivel de rentabilidad (2 alcobas):", df_final["nivel_rentabilidad_2alc"].value_counts().to_dict())
print("Todas las validaciones del enriquecimiento se cumplen.")

Participación por ciudad (debe ser 100): [100.]
Distribución de predominio: {'Equilibrado': 420, 'Arriendo': 290, 'Venta': 269}
Distribución de nivel de rentabilidad (2 alcobas): {'No comparable': 671, 'Media (6-8 %)': 219, 'Baja (<6 %)': 56, 'Alta (>=8 %)': 33}
Todas las validaciones del enriquecimiento se cumplen.


## 4. Respuestas a las preguntas de negocio

Cada tabla tiene nombres claros, unidades en el nombre de la columna y un nivel de detalle definido (barrio o ciudad). Los precios son **medianas de precios pedidos** en avisos de apartamentos al 5 de octubre de 2026.

### P1. ¿Cuánto cuesta arrendar y comprar en cada barrio?
**Nivel de detalle:** barrio. **Procedimiento:** se toman las medianas en millones de COP, se calcula el arriendo anual y los años equivalentes, y se clasifica cada barrio por cuartil dentro de su ciudad y por banda de presupuesto. Se ordena por ciudad y de mayor a menor arriendo.

In [17]:
tabla_p1 = (df_final[[  # Columnas de la tabla P1
    "ciudad", "barrio", "arriendo_mediano_mill_cop_mes", "arriendo_anual_mill_cop", "venta_mediana_mill_cop",
    "anios_arriendo_equivalentes", "rango_arriendo_ciudad", "rango_venta_ciudad",
    "banda_presupuesto_arriendo", "banda_presupuesto_venta", "nivel_confianza"]]
    .sort_values(["ciudad", "arriendo_mediano_mill_cop_mes"], ascending=[True, False])  # Ciudad y arriendo de mayor a menor
    .round({"arriendo_mediano_mill_cop_mes": 3, "arriendo_anual_mill_cop": 3, "venta_mediana_mill_cop": 2,
            "anios_arriendo_equivalentes": 1})  # Redondeo para presentación
    .reset_index(drop=True))
print("Filas de la tabla P1:", len(tabla_p1))  # Un registro por barrio
med_conf = tabla_p1[(tabla_p1["ciudad"] == "Medellín") & (tabla_p1["nivel_confianza"] == "Alta")]  # Medellín, muestra suficiente
print("Medellín: 5 barrios más caros para arrendar (muestra suficiente)")
display(med_conf.head(5))
print("Medellín: 5 barrios más económicos para arrendar (muestra suficiente)")
display(med_conf.tail(5))
print("Barrios por banda de presupuesto de arriendo y ciudad (muestra suficiente):")
pd.crosstab(df_final.loc[df_final["muestra_suficiente"], "ciudad"],
            df_final.loc[df_final["muestra_suficiente"], "banda_presupuesto_arriendo"])[["Hasta 1,5 M", "1,5 a 3 M", "3 a 5 M", "Más de 5 M"]]

Filas de la tabla P1: 979
Medellín: 5 barrios más caros para arrendar (muestra suficiente)


,ciudad,barrio,arriendo_mediano_mill_cop_mes,arriendo_anual_mill_cop,venta_mediana_mill_cop,anios_arriendo_equivalentes,rango_arriendo_ciudad,rango_venta_ciudad,banda_presupuesto_arriendo,banda_presupuesto_venta,nivel_confianza
616,Medellín,La Calera,12.0,144.0,2250.0,15.6,Alto,Alto,Más de 5 M,Más de 1.000 M,Alta
617,Medellín,La Tomatera,8.0,96.0,1595.0,16.6,Alto,Alto,Más de 5 M,Más de 1.000 M,Alta
619,Medellín,El Tesoro,7.5,90.0,1300.0,14.4,Alto,Alto,Más de 5 M,Más de 1.000 M,Alta
621,Medellín,El Campestre,7.0,84.0,1350.0,16.1,Alto,Alto,Más de 5 M,Más de 1.000 M,Alta
622,Medellín,Provenza,7.0,84.0,1300.0,15.5,Alto,Alto,Más de 5 M,Más de 1.000 M,Alta


Medellín: 5 barrios más económicos para arrendar (muestra suficiente)


,ciudad,barrio,arriendo_mediano_mill_cop_mes,arriendo_anual_mill_cop,venta_mediana_mill_cop,anios_arriendo_equivalentes,rango_arriendo_ciudad,rango_venta_ciudad,banda_presupuesto_arriendo,banda_presupuesto_venta,nivel_confianza
927,Medellín,El Salvador,1.50,18.00,335.0,18.6,Bajo,Bajo,"Hasta 1,5 M",300 a 600 M,Alta
933,Medellín,San Antonio de Prado,1.40,16.80,238.5,14.2,Bajo,Bajo,"Hasta 1,5 M",Hasta 300 M,Alta
934,Medellín,Campo Valdes,1.34,16.08,290.0,18.0,Bajo,Bajo,"Hasta 1,5 M",Hasta 300 M,Alta
937,Medellín,Las Mercedes,1.29,15.48,385.0,24.9,Bajo,Bajo,"Hasta 1,5 M",300 a 600 M,Alta
938,Medellín,Caicedo,1.29,15.48,320.0,20.7,Bajo,Bajo,"Hasta 1,5 M",300 a 600 M,Alta


Barrios por banda de presupuesto de arriendo y ciudad (muestra suficiente):


banda_presupuesto_arriendo,"Hasta 1,5 M","1,5 a 3 M",3 a 5 M,Más de 5 M
ciudad,,,,
Barranquilla,3,19,12,0
Bogotá,25,94,33,45
Bucaramanga,5,19,6,0
Cali,21,28,10,6
Cartagena,0,3,10,1
Chía,5,7,0,0
Cúcuta,3,4,0,0
Madrid,2,0,0,0
Manizales,0,1,0,0


**Consulta de presupuesto (objetivo 2).** Ejemplo: barrios de Medellín con arriendo mediano de hasta 3 millones al mes y muestra suficiente, de mayor a menor oferta. Cambia los parámetros para otra ciudad o presupuesto.

In [18]:
def barrios_por_presupuesto(ciudad: str, arriendo_max_mill: float, solo_confiables: bool = True) -> pd.DataFrame:  # Filtro de presupuesto
    """Barrios de una ciudad cuyo arriendo mediano no supera el presupuesto, ordenados por oferta."""
    filtro = (df_final["ciudad"] == ciudad) & (df_final["arriendo_mediano_mill_cop_mes"] <= arriendo_max_mill)  # Ciudad y presupuesto
    if solo_confiables:  # Opcional: solo barrios con muestra suficiente
        filtro &= df_final["muestra_suficiente"]
    columnas = ["barrio", "arriendo_mediano_mill_cop_mes", "venta_mediana_mill_cop", "avisos_arriendo", "avisos_venta", "total_avisos"]
    return df_final.loc[filtro, columnas].sort_values("total_avisos", ascending=False).reset_index(drop=True)  # Más oferta primero

consulta = barrios_por_presupuesto("Medellín", 3.0)  # Ejemplo: Medellín, hasta 3 millones al mes
print("Barrios que cumplen:", len(consulta))
consulta.head(10)

Barrios que cumplen: 100


,barrio,arriendo_mediano_mill_cop_mes,venta_mediana_mill_cop,avisos_arriendo,avisos_venta,total_avisos
0,Calasanz,2.60,440.0,2281,968,3249
1,Suramerica,2.90,555.0,423,850,1273
2,Rodeo Alto,2.40,375.0,607,367,974
3,Boston,1.85,350.0,619,328,947
4,La Doctora,3.00,520.0,435,512,947
5,San Joaquin,2.70,630.0,640,277,917
6,San Antonio de Prado,1.40,238.5,514,334,848
7,Pan de Azucar,3.00,520.0,412,384,796
8,Centro,2.30,450.0,451,333,784
9,La Aldea,2.60,492.5,537,222,759


**Respuesta P1.** La tabla `p1_costo_por_barrio.csv` da, para los 979 barrios, el arriendo mediano mensual, el arriendo anual y el precio mediano de venta. En **Medellín** (229 barrios con muestra suficiente) el arriendo mediano va de **1,29 millones/mes** (Las Mercedes y Caicedo) a **12,0 millones/mes** (La Calera), con una mediana entre barrios de **3,2 millones**; la venta va de 238,5 a 2.250 millones. En **Bogotá** (197 barrios) el arriendo va de 1,10 a 14,5 millones/mes (mediana 2,6). Para presupuestar: en Medellín la mayoría de barrios confiables está entre **1,5 y 3 millones** (94) o entre **3 y 5 millones** (93); solo 6 barrios confiables cuestan hasta 1,5 millones. La consulta de ejemplo encuentra **100 barrios** de Medellín con arriendo de hasta 3 millones y muestra suficiente; el de mayor oferta es Calasanz (2,6 millones/mes, 3.249 avisos). El rango relativo indica que en Medellín un barrio es de costo **Alto** desde unos 4,2 millones/mes y en Bogotá desde unos 3,6 millones.

**Utilidad:** permite saber cuánto presupuestar por barrio y encontrar barrios dentro de un presupuesto (objetivos 1 y 2).

**Limitaciones:** son medianas de precios pedidos de apartamentos de cualquier tamaño, no precios de cierre; un barrio con apartamentos más grandes aparecerá más caro aunque el m² no lo sea (el dataset no trae área). Los 359 barrios con nivel de confianza Baja (menos de 30 avisos en arriendo o en venta) deben leerse como referencia.

### P2. ¿Cuántas ofertas de arriendo y de venta tiene cada barrio y cuál predomina?
**Nivel de detalle:** barrio (tabla principal) y ciudad (resumen de predominio). **Procedimiento:** se suman los avisos, se calcula el porcentaje en arriendo y se clasifica el predominio con la regla 60/40.

In [19]:
tabla_p2 = (df_final[["ciudad", "barrio", "avisos_arriendo", "avisos_venta", "total_avisos",
                      "pct_avisos_arriendo", "predominio_oferta", "nivel_confianza"]]
            .sort_values(["ciudad", "total_avisos"], ascending=[True, False])  # Ciudad y oferta de mayor a menor
            .round({"pct_avisos_arriendo": 1})
            .reset_index(drop=True))
tabla_p2_predominio = (pd.crosstab(df_final["ciudad"], df_final["predominio_oferta"], margins=True, margins_name="Total")  # Barrios por predominio
                       [["Arriendo", "Equilibrado", "Venta", "Total"]]
                       .rename_axis(index="ciudad", columns=None).reset_index())
assert tabla_p2_predominio.loc[tabla_p2_predominio["ciudad"] == "Total", "Total"].item() == len(df_final)  # Cuadra con el total de barrios
display(tabla_p2[tabla_p2["ciudad"] == "Medellín"].head(10))  # Vista: Medellín
tabla_p2_predominio

,ciudad,barrio,avisos_arriendo,avisos_venta,total_avisos,pct_avisos_arriendo,predominio_oferta,nivel_confianza
616,Medellín,Laureles,6722,4067,10789,62.3,Arriendo,Alta
617,Medellín,Castropol,2788,666,3454,80.7,Arriendo,Alta
618,Medellín,El Tesoro,2680,719,3399,78.8,Arriendo,Alta
619,Medellín,Calasanz,2281,968,3249,70.2,Arriendo,Alta
620,Medellín,Santa Maria de los Angeles,2321,649,2970,78.1,Arriendo,Alta
621,Medellín,Las Palmas,1941,672,2613,74.3,Arriendo,Alta
622,Medellín,Ciudad del Rio,2008,511,2519,79.7,Arriendo,Alta
623,Medellín,Altos del Poblado,1825,616,2441,74.8,Arriendo,Alta
624,Medellín,Los Balsos,2002,286,2288,87.5,Arriendo,Alta
625,Medellín,Loma de los Bernal,1314,887,2201,59.7,Equilibrado,Alta


,ciudad,Arriendo,Equilibrado,Venta,Total
0,Barranquilla,20,18,9,47
1,Bogotá,49,152,132,333
2,Bucaramanga,7,36,9,52
3,Cali,16,43,56,115
4,Cartagena,1,10,8,19
5,Chía,2,9,12,23
6,Cúcuta,13,4,0,17
7,Madrid,0,1,5,6
8,Manizales,3,1,0,4
9,Medellín,162,128,34,324


**Respuesta P2.** La tabla `p2_ofertas_por_barrio.csv` muestra para cada barrio cuántos avisos hay en arriendo y en venta. En total hay **282.591 avisos** (155.355 de arriendo y 127.236 de venta). El predominio cambia mucho entre ciudades: en **Medellín** el **65,2 %** de los avisos son de arriendo y **162 de sus 324 barrios** tienen predominio de arriendo (solo 34 de venta); en **Bogotá** ocurre lo contrario: el arriendo es el **41,9 %** de la oferta y hay **132 barrios** con predominio de venta frente a 49 de arriendo. En el país, 420 barrios están equilibrados (40 % a 60 % de arriendo), 290 tienen predominio de arriendo y 269 de venta.

**Utilidad:** quien busca arrendar tiene más opciones en barrios con predominio de arriendo; quien busca comprar, en los de predominio de venta (objetivo 2).

**Limitaciones:** el número de avisos mide la oferta publicada en la fecha de corte, no los negocios cerrados ni la demanda; un mismo inmueble puede estar publicado en arriendo y en venta.

### P3. ¿Qué barrios concentran más oferta en cada ciudad?
**Nivel de detalle:** barrio dentro de ciudad (10 primeros puestos). **Procedimiento:** se ordena por total de avisos, se toma el top 10 de cada ciudad y se calcula la participación individual y acumulada sobre la oferta total de la ciudad (todos sus barrios).

In [20]:
tabla_p3 = (df_final[df_final["puesto_oferta_ciudad"] <= 10]  # Top 10 de cada ciudad
            .sort_values(["ciudad", "puesto_oferta_ciudad"])
            [["ciudad", "puesto_oferta_ciudad", "barrio", "avisos_arriendo", "avisos_venta", "total_avisos",
              "participacion_oferta_ciudad_pct", "arriendo_mediano_mill_cop_mes", "venta_mediana_mill_cop", "nivel_confianza"]]
            .reset_index(drop=True))
tabla_p3["participacion_acumulada_pct"] = tabla_p3.groupby("ciudad")["participacion_oferta_ciudad_pct"].cumsum()  # Acumulado del top
tabla_p3 = tabla_p3.round({"participacion_oferta_ciudad_pct": 2, "participacion_acumulada_pct": 2,
                           "arriendo_mediano_mill_cop_mes": 3, "venta_mediana_mill_cop": 2})
assert (tabla_p3["participacion_acumulada_pct"] <= 100 + 1e-6).all()  # El acumulado nunca supera 100 %
display(tabla_p3[tabla_p3["ciudad"] == "Medellín"])  # Top 10 de Medellín
tabla_p3[tabla_p3["ciudad"] == "Bogotá"]  # Top 10 de Bogotá

,ciudad,puesto_oferta_ciudad,barrio,avisos_arriendo,avisos_venta,total_avisos,participacion_oferta_ciudad_pct,arriendo_mediano_mill_cop_mes,venta_mediana_mill_cop,nivel_confianza,participacion_acumulada_pct
80,Medellín,1,Laureles,6722,4067,10789,8.16,4.00,791.0,Alta,8.16
81,Medellín,2,Castropol,2788,666,3454,2.61,5.35,980.0,Alta,10.77
82,Medellín,3,El Tesoro,2680,719,3399,2.57,7.50,1300.0,Alta,13.34
83,Medellín,4,Calasanz,2281,968,3249,2.46,2.60,440.0,Alta,15.80
84,Medellín,5,Santa Maria de los Angeles,2321,649,2970,2.25,5.50,1050.0,Alta,18.04
85,Medellín,6,Las Palmas,1941,672,2613,1.98,4.30,945.0,Alta,20.02
86,Medellín,7,Ciudad del Rio,2008,511,2519,1.90,4.50,790.0,Alta,21.92
87,Medellín,8,Altos del Poblado,1825,616,2441,1.85,5.50,1100.0,Alta,23.77
88,Medellín,9,Los Balsos,2002,286,2288,1.73,6.00,1375.0,Alta,25.50
89,Medellín,10,Loma de los Bernal,1314,887,2201,1.66,3.30,645.0,Alta,27.16


,ciudad,puesto_oferta_ciudad,barrio,avisos_arriendo,avisos_venta,total_avisos,participacion_oferta_ciudad_pct,arriendo_mediano_mill_cop_mes,venta_mediana_mill_cop,nivel_confianza,participacion_acumulada_pct
10,Bogotá,1,Chico Norte,1958,1832,3790,4.64,5.50,960.00,Alta,4.64
11,Bogotá,2,Chico,1173,1535,2708,3.31,7.00,1200.00,Alta,7.95
12,Bogotá,3,Santa Barbara,722,1555,2277,2.79,4.50,920.00,Alta,10.74
13,Bogotá,4,Cedritos,967,1229,2196,2.69,2.75,550.00,Alta,13.43
14,Bogotá,5,Chico Reservado,1044,1045,2089,2.56,9.50,1450.00,Alta,15.99
15,Bogotá,6,Los Rosales,855,1102,1957,2.40,10.50,1850.00,Alta,18.38
16,Bogotá,7,La Cabrera,1093,803,1896,2.32,13.00,2350.00,Alta,20.70
17,Bogotá,8,Santa Barbara Central,522,1253,1775,2.17,5.00,900.00,Alta,22.88
18,Bogotá,9,Santa Barbara Occidental,444,1124,1568,1.92,4.80,980.00,Alta,24.79
19,Bogotá,10,Santa Paula,505,978,1483,1.82,5.50,1032.95,Alta,26.61


**Respuesta P3.** La tabla `p3_top10_oferta_por_ciudad.csv` ordena los 10 barrios con más avisos de cada ciudad. **Laureles (Medellín)** es el barrio con más oferta del país: **10.789 avisos**, el **8,16 %** de la oferta de Medellín y casi tres veces el segundo barrio del país, Chico Norte (Bogotá, 3.790 avisos, 4,64 % de su ciudad). Le siguen en Medellín Castropol (3.454), El Tesoro (3.399) y Calasanz (3.249). Los 10 primeros barrios concentran el **27,2 %** de la oferta en Medellín y el **26,6 %** en Bogotá: la oferta está repartida en muchos barrios. En ciudades más pequeñas la concentración es mucho mayor (Rionegro 72,9 %, Cartagena 83,0 %).

**Utilidad:** identifica dónde hay más opciones para buscar y negociar (objetivo 3).

**Limitaciones:** el top incluye barrios de muy distinto precio (Laureles 4,0 millones/mes frente a Calasanz 2,6); debe cruzarse con la tabla P1 según el presupuesto. En ciudades con pocos barrios el top 10 cubre casi toda la ciudad.

### P4. ¿En qué barrios conviene más arrendar o comprar?
**Nivel de detalle:** barrio comparable. **Procedimiento:** se usan solo los barrios con `comparable_2alc` (≥ 20 avisos de apartamentos de 2 alcobas en arriendo y en venta), que es el criterio del autor para comparar inmuebles parecidos. La rentabilidad bruta anual = arriendo × 12 ÷ precio de venta. Para quien **invierte**, más rentabilidad es mejor negocio de arriendo; para quien **busca vivienda**, una rentabilidad baja significa que arrendar sale relativamente barato frente a comprar.

In [21]:
tabla_p4 = (df_final[df_final["comparable_2alc"]]  # Solo barrios comparables
            [["ciudad", "barrio", "arriendo_2alc_mill_cop_mes", "venta_2alc_mill_cop", "rentabilidad_bruta_2alc_pct",
              "anios_arriendo_2alc", "nivel_rentabilidad_2alc", "rentabilidad_2alc_atipica"]]
            .sort_values(["ciudad", "rentabilidad_bruta_2alc_pct"], ascending=[True, False])  # Más rentable primero
            .round({"arriendo_2alc_mill_cop_mes": 3, "venta_2alc_mill_cop": 2, "rentabilidad_bruta_2alc_pct": 2, "anios_arriendo_2alc": 1})
            .reset_index(drop=True))
print("Barrios comparables:", len(tabla_p4))
print(tabla_p4["nivel_rentabilidad_2alc"].value_counts().to_string())
med4 = tabla_p4[(tabla_p4["ciudad"] == "Medellín") & ~tabla_p4["rentabilidad_2alc_atipica"]]  # Medellín sin atípicos
print("Medellín: mayor rentabilidad (mejor para invertir)")
display(med4.head(5))
print("Medellín: menor rentabilidad (arrendar sale relativamente más barato que comprar)")
med4.tail(5)

Barrios comparables: 308
nivel_rentabilidad_2alc
Media (6-8 %)    219
Baja (<6 %)       56
Alta (>=8 %)      33
Medellín: mayor rentabilidad (mejor para invertir)


,ciudad,barrio,arriendo_2alc_mill_cop_mes,venta_2alc_mill_cop,rentabilidad_bruta_2alc_pct,anios_arriendo_2alc,nivel_rentabilidad_2alc,rentabilidad_2alc_atipica
169,Medellín,San Rafael,3.2,430.0,8.93,11.2,Alta (>=8 %),False
170,Medellín,Loma del Indio,3.7,530.0,8.38,11.9,Alta (>=8 %),False
171,Medellín,Calasanz,2.4,355.0,8.11,12.3,Alta (>=8 %),False
172,Medellín,Santa Rosa de Lima,2.4,357.5,8.06,12.4,Alta (>=8 %),False
173,Medellín,Las Acacias,3.1,465.0,8.00,12.5,Alta (>=8 %),False


Medellín: menor rentabilidad (arrendar sale relativamente más barato que comprar)


,ciudad,barrio,arriendo_2alc_mill_cop_mes,venta_2alc_mill_cop,rentabilidad_bruta_2alc_pct,anios_arriendo_2alc,nivel_rentabilidad_2alc,rentabilidad_2alc_atipica
280,Medellín,El Campestre,5.95,1300.0,5.49,18.2,Baja (<6 %),False
281,Medellín,La Florida,4.50,1030.0,5.24,19.1,Baja (<6 %),False
282,Medellín,La Castellana,2.80,650.0,5.17,19.3,Baja (<6 %),False
283,Medellín,Transversal Inferior,4.60,1100.0,5.02,19.9,Baja (<6 %),False
284,Medellín,Astorga,4.25,1050.0,4.86,20.6,Baja (<6 %),False


**Respuesta P4.** La tabla `p4_arrendar_vs_comprar.csv` compara **308 barrios** con suficientes apartamentos de 2 alcobas en arriendo y en venta. La rentabilidad bruta mediana es **6,77 %**, es decir, un apartamento cuesta en promedio el equivalente a **14,8 años de arriendo**. 219 barrios tienen rentabilidad Media (6 % a 8 %), 33 Alta (8 % o más) y 56 Baja (menos de 6 %). En **Medellín** (121 barrios, mediana 6,64 %), sin contar atípicos, los más rentables para invertir son **San Rafael (8,93 %, 11,2 años)**, Loma del Indio (8,38 %) y Calasanz (8,11 %); donde arrendar sale relativamente más barato que comprar son **Astorga (4,86 %, 20,6 años)**, Transversal Inferior (5,02 %) y La Castellana (5,17 %). Laureles, el barrio con más oferta, está en 6,53 % (15,3 años).

**Utilidad:** orienta la decisión de comprar para arrendar (inversionista) o de arrendar en lugar de comprar (quien busca vivienda), que es el "valor de hacer el negocio" del objetivo general.

**Limitaciones:** es rentabilidad **bruta** e **indicativa**: las medianas de arriendo y de venta vienen de avisos distintos, no del mismo apartamento, y no descuenta administración, impuestos, vacancia ni comisiones. 24 barrios con valores atípicos (por ejemplo La Inmaculada, Medellín, 16,27 %) están marcados y no se usan en los ejemplos.

### P5. ¿Cómo se comparan las ciudades en costo y oferta?
**Nivel de detalle:** ciudad. **Procedimiento:** se agregan los avisos de todos los barrios; los precios de la ciudad son la **mediana entre barrios con muestra suficiente**, y la rentabilidad, la mediana entre barrios comparables de 2 alcobas. Las ciudades con menos de 10 barrios confiables se marcan como referenciales.

In [22]:
conf = df_final[df_final["muestra_suficiente"]]  # Barrios con muestra suficiente
comp = df_final[df_final["comparable_2alc"]]  # Barrios comparables (2 alcobas)
tabla_p5 = (df_final.groupby("ciudad").agg(  # Oferta: todos los barrios
                barrios=("barrio", "size"),
                avisos_arriendo=("avisos_arriendo", "sum"),
                avisos_venta=("avisos_venta", "sum"),
                total_avisos=("total_avisos", "sum"))
            .join(conf.groupby("ciudad").agg(  # Precios: solo barrios confiables
                barrios_muestra_suficiente=("barrio", "size"),
                arriendo_mediano_entre_barrios_mill_cop_mes=("arriendo_mediano_mill_cop_mes", "median"),
                venta_mediana_entre_barrios_mill_cop=("venta_mediana_mill_cop", "median")))
            .join(comp.groupby("ciudad").agg(  # Rentabilidad: barrios comparables
                barrios_comparables_2alc=("barrio", "size"),
                rentabilidad_mediana_2alc_pct=("rentabilidad_bruta_2alc_pct", "median")))
            .join(df_final[df_final["puesto_oferta_ciudad"] <= 10].groupby("ciudad")["participacion_oferta_ciudad_pct"]
                  .sum().rename("concentracion_top10_pct")))  # % de la oferta en sus 10 barrios principales
tabla_p5[["barrios_muestra_suficiente", "barrios_comparables_2alc"]] = tabla_p5[["barrios_muestra_suficiente", "barrios_comparables_2alc"]].fillna(0).astype(int)
tabla_p5["participacion_oferta_nacional_pct"] = tabla_p5["total_avisos"] / tabla_p5["total_avisos"].sum() * 100  # Peso de la ciudad
tabla_p5["alcance"] = np.where(tabla_p5["barrios_muestra_suficiente"] >= 10, "Suficiente", "Referencial (<10 barrios confiables)")
tabla_p5 = (tabla_p5.sort_values("total_avisos", ascending=False).reset_index()
            .round({"arriendo_mediano_entre_barrios_mill_cop_mes": 3, "venta_mediana_entre_barrios_mill_cop": 2,
                    "rentabilidad_mediana_2alc_pct": 2, "concentracion_top10_pct": 2, "participacion_oferta_nacional_pct": 2}))

assert tabla_p5["total_avisos"].sum() == df_final["total_avisos"].sum()  # Totales cuadran con el dataset
assert tabla_p5["barrios"].sum() == len(df_final)  # Todos los barrios incluidos
assert abs(tabla_p5["participacion_oferta_nacional_pct"].sum() - 100) < 0.05  # Participaciones suman 100 % (redondeo)
ref = tabla_p5.set_index("ciudad")["rentabilidad_mediana_2alc_pct"]  # Validación externa con la ficha del dataset
for ciudad, valor_ficha in {"Bogotá": 7.0, "Medellín": 6.6, "Cartagena": 6.0, "Barranquilla": 7.7}.items():
    assert abs(ref[ciudad] - valor_ficha) < 0.05, (ciudad, ref[ciudad])  # La ficha reporta estos valores redondeados
print("Totales, participaciones y rentabilidades coinciden con el dataset y con su ficha oficial.")
tabla_p5

Totales, participaciones y rentabilidades coinciden con el dataset y con su ficha oficial.


,ciudad,barrios,avisos_arriendo,avisos_venta,total_avisos,barrios_muestra_suficiente,arriendo_mediano_entre_barrios_mill_cop_mes,venta_mediana_entre_barrios_mill_cop,barrios_comparables_2alc,rentabilidad_mediana_2alc_pct,concentracion_top10_pct,participacion_oferta_nacional_pct,alcance
0,Medellín,324,86273,45986,132259,229,3.200,590.0,121,6.64,27.16,46.80,Suficiente
1,Bogotá,333,34248,47448,81696,197,2.600,580.0,86,6.97,26.61,28.91,Suficiente
2,Cali,115,9980,14451,24431,65,1.700,400.0,37,6.96,38.80,8.65,Suficiente
3,Rionegro,33,8814,4842,13656,24,2.600,537.5,21,6.09,72.86,4.83,Suficiente
4,Barranquilla,47,5792,4126,9918,34,2.550,400.0,17,7.71,51.41,3.51,Suficiente
5,Bucaramanga,52,3819,4012,7831,30,2.236,472.5,5,5.88,52.39,2.77,Suficiente
6,Cartagena,19,2309,3316,5625,14,4.000,754.4,11,6.00,82.97,1.99,Suficiente
7,Cúcuta,17,1985,699,2684,7,1.600,350.0,0,NaN,82.49,0.95,Referencial (<10 barrios confiables)
8,Chía,23,850,1387,2237,12,1.630,319.0,9,6.49,65.18,0.79,Suficiente
9,Soacha,4,390,487,877,4,0.775,154.5,1,6.62,100.00,0.31,Referencial (<10 barrios confiables)


**Respuesta P5.** La tabla `p5_resumen_por_ciudad.csv` resume las 13 ciudades. **Medellín** concentra el **46,8 %** de la oferta del dataset (132.259 avisos) y **Bogotá** el **28,9 %**. Entre barrios confiables, el arriendo mediano es más alto en **Cartagena (4,0 millones/mes)** y **Medellín (3,2)** que en Bogotá (2,6), Cali (1,7) o Barranquilla (2,55); la venta mediana es más alta en Cartagena (754,4 millones), Medellín (590) y Bogotá (580). La rentabilidad mediana de 2 alcobas es mayor en **Barranquilla (7,71 %)** y Bogotá (6,97 %) y menor en Bucaramanga (5,88 %), Cartagena (6,00 %) y Rionegro (6,09 %). Estas rentabilidades coinciden con las publicadas en la ficha del dataset (7,0 % Bogotá; 6,6 % Medellín; 6,0 % Cartagena; 7,7 % Barranquilla), lo que valida el procesamiento.

**Utilidad:** permite aplicar el informe a otras ciudades y saber en cuáles hay información suficiente.

**Limitaciones:** Cúcuta, Soacha, Pereira, Madrid y Manizales tienen menos de 10 barrios confiables y se marcan como **referenciales**; en 4 de ellas no hay barrios comparables de 2 alcobas, por lo que no se calcula su rentabilidad.

## 5. Exportación y verificación de archivos
Se exportan el dataset preparado, las tablas y la bitácora en CSV (UTF-8). La función `exportar_csv_verificado` vuelve a leer cada archivo y comprueba que coincida con la tabla del notebook; si algo difiere, la celda falla.

In [23]:
exportaciones = {  # Ruta de destino -> tabla
    RUTA_DATASET: df_final,
    CARPETA_TABLAS / "p1_costo_por_barrio.csv": tabla_p1,
    CARPETA_TABLAS / "p2_ofertas_por_barrio.csv": tabla_p2,
    CARPETA_TABLAS / "p2_predominio_por_ciudad.csv": tabla_p2_predominio,
    CARPETA_TABLAS / "p3_top10_oferta_por_ciudad.csv": tabla_p3,
    CARPETA_TABLAS / "p4_arrendar_vs_comprar.csv": tabla_p4,
    CARPETA_TABLAS / "p5_resumen_por_ciudad.csv": tabla_p5,
    CARPETA_TABLAS / "control_bitacora_calidad.csv": tabla_bitacora,
    CARPETA_TABLAS / "control_comparacion_estructura.csv": comparar_estructura(df_original, df_final, nombres),
}
registro = []  # Lista de archivos exportados
for ruta, tabla in exportaciones.items():  # Exportamos y verificamos cada tabla
    exportar_csv_verificado(tabla, ruta)  # Falla si el CSV no coincide con la tabla
    registro.append({"archivo": ruta.relative_to(ROOT).as_posix(), "filas": len(tabla), "columnas": tabla.shape[1]})
pd.DataFrame(registro)  # Archivos generados y su tamaño

,archivo,filas,columnas
0,AE3/datos/dataset_preparado.csv,979,36
1,AE3/resultados/tablas/p1_costo_por_barrio.csv,979,11
2,AE3/resultados/tablas/p2_ofertas_por_barrio.csv,979,8
3,AE3/resultados/tablas/p2_predominio_por_ciudad...,14,5
4,AE3/resultados/tablas/p3_top10_oferta_por_ciud...,106,11
5,AE3/resultados/tablas/p4_arrendar_vs_comprar.csv,308,8
6,AE3/resultados/tablas/p5_resumen_por_ciudad.csv,13,13
7,AE3/resultados/tablas/control_bitacora_calidad...,15,5
8,AE3/resultados/tablas/control_comparacion_estr...,38,7


### 5.1 Resumen descriptivo del DataFrame final (.txt)
Se genera con la clase `Reportes` (trabajada en Triple A) a partir de `df_final`, el mismo DataFrame exportado como dataset preparado. Se sobrescribe en cada ejecución.

In [24]:
metadatos = {  # Información de contexto que encabeza el reporte
    "Generado": f"{datetime.now():%Y-%m-%d %H:%M:%S}",
    "Dataset": DATASET_ID,
    "URL": f"https://huggingface.co/datasets/{DATASET_ID}",
    "Configuración": CONFIG,
    "Split": SPLIT,
    "Revisión": REVISION,
    "Muestra": "No (dataset completo)",
    "Fecha de corte de los datos": f"{df_final['fecha_corte'].min():%Y-%m-%d}",
    "Filas originales -> finales": f"{len(df_original)} -> {len(df_final)}",
    "Columnas originales -> finales": f"{df_original.shape[1]} -> {df_final.shape[1]}",
    "Unidades": "precios en millones de COP (arriendo por mes); porcentajes en %",
    "Archivo de datos": RUTA_DATASET.relative_to(ROOT).as_posix(),
}
reportes = Reportes()  # Clase de reportes de src/clases_po
reportes.guardar_reporte(df_final, RUTA_TXT, "RESUMEN DEL DATAFRAME TRANSFORMADO Y ENRIQUECIDO (AE3)", metadatos)  # Crea carpeta y escribe en UTF-8

contenido = RUTA_TXT.read_text(encoding="utf-8")  # Verificamos el archivo recién escrito
assert f"Filas: {len(df_final)}" in contenido and f"Columnas: {df_final.shape[1]}" in contenido  # Corresponde al DataFrame final
assert all(f"- {c}:" in contenido for c in df_final.columns)  # Incluye todas las columnas
print(contenido)  # Mostramos el resumen completo

RESUMEN DEL DATAFRAME TRANSFORMADO Y ENRIQUECIDO (AE3)
Generado: 2026-11-08 14:39:42
Dataset: aptologia/arriendo-rentabilidad-por-barrio-colombia-2026
URL: https://huggingface.co/datasets/aptologia/arriendo-rentabilidad-por-barrio-colombia-2026
Configuración: rentabilidad_arriendo_venta_barrio
Split: train
Revisión: b752ea67e4c2615a723900d7654e3553e6a546c1
Muestra: No (dataset completo)
Fecha de corte de los datos: 2026-10-05
Filas originales -> finales: 979 -> 979
Columnas originales -> finales: 19 -> 36
Unidades: precios en millones de COP (arriendo por mes); porcentajes en %
Archivo de datos: AE3/datos/dataset_preparado.csv

Filas: 979
Columnas: 36
Duplicados: 0

Columnas y tipos:
- fecha_corte: datetime64[us]
- ciudad_id: str
- ciudad: str
- barrio_id: str
- barrio: str
- barrio_clave: str
- avisos_arriendo: int64
- avisos_venta: int64
- arriendo_mediano_mill_cop_mes: float64
- venta_mediana_mill_cop: float64
- rentabilidad_bruta_pct: float64
- muestra_suficiente: bool
- rentabilid

## 6. Conclusiones del avance

1. **Los datos quedaron preparados sin perder información:** los 979 barrios se conservaron; los 15 controles de calidad no encontraron duplicados, inconsistencias de texto ni valores fuera de rango. Los 1.860 nulos son estructurales (muestra insuficiente de 2 alcobas) y no se imputaron. Los 60 barrios con rentabilidad atípica se marcaron en vez de eliminarse.
2. **Las cinco preguntas tienen respuesta sustentada en una tabla exportada.** Medellín es la ciudad con más oferta y con predominio de arriendo; Bogotá tiene más oferta en venta; Laureles es el barrio con más avisos del país; un apartamento equivale en mediana a 14,8 años de arriendo.
3. **Validación:** los totales y las participaciones cuadran, cada CSV coincide con su tabla y las rentabilidades por ciudad coinciden con la ficha oficial del dataset.
4. **Pendiente para AE4:** representar gráficamente las tablas (mapa de barrios por costo, ranking de oferta, dispersión arriendo-venta). **Quedan sin resolver**, por falta de variables: precio por m², estrato, costos de administración y evolución en el tiempo (los datos son de una sola fecha).